# 4. Vanna Training

This notebook trains Vanna with the schema and sample queries for the EMR database.
It dynamically loads training data from version-controlled files under the `vanna_training` folder.

In [4]:
import os
import sys
import yaml

# Ultimate path injection to find 'src' regardless of current working directory
cwd = os.path.abspath(os.getcwd())
project_root = None

# Walk up from current working directory
temp = cwd
for _ in range(4):
    if os.path.exists(os.path.join(temp, 'src', 'config.py')):
        project_root = temp
        break
    parent = os.path.abspath(os.path.join(temp, '..'))
    if parent == temp:
        break
    temp = parent

if project_root:
    if project_root not in sys.path:
        sys.path.insert(0, project_root)
    path_prefix = os.path.relpath(project_root, cwd)
    print(f"Project root found: {project_root}")
else:
    print("Warning: Could not automatically detect project root. Using default path prefix '..'")
    path_prefix = ".."
    project_root = os.path.abspath(path_prefix)

from src.services.providers import get_vanna

Project root found: d:\PEMROGRAMAN\LLM-DESKTOP\local-rag-comparator


In [5]:
print("1. Initializing Vanna...")
vn = get_vanna()

# Reset training collections so stale embeddings from older runs do not conflict
for collection_name in ["sql", "ddl", "documentation"]:
    try:
        vn.remove_collection(collection_name)
        print(f"Reset collection: {collection_name}")
    except Exception as e:
        print(f"Skipping reset for {collection_name}: {e}")

1. Initializing Vanna...
Reset collection: sql
Reset collection: ddl
Reset collection: documentation


In [6]:
print("2. Training Vanna with DDL...")
schema_path = os.path.join(project_root, "vanna_training", "schema.sql")
print(f"Loading schema from: {schema_path}")
with open(schema_path, "r", encoding="utf-8") as f:
    ddl = f.read()
vn.train(ddl=ddl)

2. Training Vanna with DDL...
Loading schema from: d:\PEMROGRAMAN\LLM-DESKTOP\local-rag-comparator\vanna_training\schema.sql
Adding ddl: CREATE TABLE emr_records (
    status VARCHAR,
    emr_name VARCHAR,
    serial_number VARCHAR,
    machine_product VARCHAR,
    machine_model VARCHAR,
    account_account_name VARCHAR,
    sub_call_type VARCHAR,
    pmact_type VARCHAR,
    subjects TEXT,
    symptom TEXT,
    caused_of_problem TEXT,
    action_how_was_problem_corrected TEXT,
    branch_site VARCHAR,
    part_suply VARCHAR,
    main_cause_part_no VARCHAR,
    part_description VARCHAR,
    part_description_1 VARCHAR,
    techcare_component VARCHAR,
    techcare_component_1 VARCHAR,
    techcare_sub_component VARCHAR,
    techcare_sub_component_1 VARCHAR,
    created_date TIMESTAMP,
    emr_last_closed_date TIMESTAMP,
    model_family VARCHAR,
    graph_community_summary TEXT
);



'f66c7e5f-e765-5c14-a84a-9359bb87fd1c-ddl'

In [7]:
print("3. Training Vanna with Documentation...")
doc_path = os.path.join(project_root, "vanna_training", "domain_docs.md")
print(f"Loading documentation from: {doc_path}")
with open(doc_path, "r", encoding="utf-8") as f:
    doc = f.read()
vn.train(documentation=doc)

3. Training Vanna with Documentation...
Loading documentation from: d:\PEMROGRAMAN\LLM-DESKTOP\local-rag-comparator\vanna_training\domain_docs.md
Adding documentation....


'cdf0cc70-0f94-5a8f-925f-81e4b3e0e5b4-doc'

In [8]:
print("4. Training Vanna with Sample Q&A...")
qa_path = os.path.join(project_root, "vanna_training", "qa_pairs.yaml")
print(f"Loading QA pairs from: {qa_path}")
with open(qa_path, "r", encoding="utf-8") as f:
    qa_data = yaml.safe_load(f)

for item in qa_data:
    q = item["question"]
    sql = item["sql"]
    vn.train(question=q, sql=sql)

print("Training complete.")

4. Training Vanna with Sample Q&A...
Loading QA pairs from: d:\PEMROGRAMAN\LLM-DESKTOP\local-rag-comparator\vanna_training\qa_pairs.yaml
Training complete.


In [9]:
print("5. Testing SQL Generation...")
test_query = "Hitung berapa banyak kasus kerusakan yang berkaitan dengan kluster Sistem Pendingin (Cooling System) menurut GraphRAG"
sql = vn.generate_sql(test_query)
print(f"Question: {test_query}\nGenerated SQL: {sql}")

if sql:
    df = vn.run_sql(sql)
    print(df)

5. Testing SQL Generation...
Question: Hitung berapa banyak kasus kerusakan yang berkaitan dengan kluster Sistem Pendingin (Cooling System) menurut GraphRAG
Generated SQL: SELECT COUNT(*) FROM emr_records WHERE graph_community_summary ILIKE '%Cooling System%' OR graph_community_summary ILIKE '%Sistem Pendingin%';
   count
0    292
